# ==================== 阶段一：EDA（三连看） ====================
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

# ① 数值列分布（看异常值）
axes[0].hist(df['数值列'], bins=30, color=蓝, edgecolor='k')
axes[0].set_title('数值列 分布')

# ② 相关性热力图
num_cols = df.select_dtypes(include='number').columns
corr = df[num_cols].corr()
im = axes[1].imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        axes[1].text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center')
axes[1].set_title('相关性矩阵')

# ③ 类别列分布（看缺失+不平衡）
vc = df['类别列'].value_counts(dropna=False)
axes[2].bar([str(k) for k in vc.index], vc.values, color=橙)
axes[2].set_title('类别分布（含缺失）')

plt.tight_layout(); plt.show()

# ==================== 阶段二：数据清洗（四连鞭） ====================
clean = df.copy()

# ① 强制转类型
clean['数值列'] = pd.to_numeric(clean['数值列'], errors='coerce')

# ② 异常值截断
lo, hi = clean['数值列'].quantile([0.01, 0.99])
clean['数值列'] = clean['数值列'].clip(lo, hi)

# ③ 缺失值填充
clean['数值列'] = clean['数值列'].fillna(clean['数值列'].median())
clean['类别列'] = clean['类别列'].fillna(clean['类别列'].mode()[0])

# ④ 删除重复行
clean = clean.drop_duplicates().reset_index(drop=True)

# ==================== 阶段三：验证 ====================
print("缺失值：\n", clean.isna().sum())
print("清理后 shape：", clean.shape)